# HM Land Registry Price Paid Data & the new UPRN look-up · **Bronze layer**

**What this notebook does, in one breath:** on 28 August 2026 HM Land Registry
published, for the first time in thirty years of Price Paid Data, a stable
property identifier — a monthly CSV pairing the transaction unique identifier
with a UPRN. This notebook downloads that file and the Price Paid file it
belongs to, fingerprints both, works out which sales the look-up is actually
*about*, and captures a figure from HMLR I can later reconcile my own extraction
against.

**Rules for Bronze:** no renaming, no fixing, no maths. An honest copy, with a
checksum and a source URL beside it.

**One rule specific to this project:** HMLR keeps only the *latest* month of
these files, at a stable URL, overwritten in place on the 20th working day of
each month. There is no archive. If I don't capture this month now, this month
is gone — so Bronze here isn't just good practice, it's the only copy that will
ever exist.

## Step 0 — Tools I need

`requests` to download, `hashlib` to fingerprint each file, and `pandas` further
down for one question I can only answer by actually looking at the data.

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

print("Tools loaded OK")

Tools loaded OK


## Step 1 — Point at my folders

Same project-root trick I use in every notebook: if Jupyter's current folder is
`notebooks/`, step up one level so the paths work whether I run this from the
notebook or from a terminal.

I key Bronze by *release* rather than by file, because these two CSVs are one
publication event. Next month's is a different folder.

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RELEASE_SLUG = "release_2026-08-28"
BRONZE_DIR = PROJECT_DIR / "data" / "bronze" / RELEASE_SLUG
SILVER_DIR = PROJECT_DIR / "data" / "silver" / RELEASE_SLUG
GOLD_DIR = PROJECT_DIR / "data" / "gold" / RELEASE_SLUG

print("Project:", PROJECT_DIR)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
print("Bronze :", BRONZE_DIR)

Project: /Users/yusufismail/hmlr-price-paid-uprn-pipeline
Bronze : /Users/yusufismail/hmlr-price-paid-uprn-pipeline/data/bronze/release_2026-08-28


## Step 2 — What actually shipped

Two files, published together at the same moment. There are a couple of traps in
the naming that cost me time, so they're worth writing down:

- The filename is **hyphenated**, not underscored: `pp-uprn-lookup-jul-2026.csv`.
- The month in the filename is the **data** month, not the publication month.
  The file published on 28 August 2026 is called `jul-2026`. Asking for
  `pp-uprn-lookup-aug-2026.csv` returns a 403 — it doesn't exist yet.
- **Neither file has a header row.** The field order comes from the spec, and I
  apply it in Silver, not here.

I'm taking the monthly update file, not the 5.1GB complete file. Step 4 shows
why that's the right denominator.

In [3]:
BASE_URL = "https://price-paid-data.publicdata.landregistry.gov.uk"

SOURCES = {
    "pp-monthly-update-new-version.csv": (
        "Price Paid Data monthly update, 'new version' format — 16 fields, "
        "including the Record Status column the complete file doesn't have."
    ),
    "pp-uprn-lookup-jul-2026.csv": (
        "Transaction unique identifier and UPRN Look Up Table — two fields, "
        "no header: the 38-character braced identifier, and a UPRN."
    ),
}

for filename, description in SOURCES.items():
    print(f"{filename}\n  {BASE_URL}/{filename}\n  {description}\n")

pp-monthly-update-new-version.csv
  https://price-paid-data.publicdata.landregistry.gov.uk/pp-monthly-update-new-version.csv
  Price Paid Data monthly update, 'new version' format — 16 fields, including the Record Status column the complete file doesn't have.

pp-uprn-lookup-jul-2026.csv
  https://price-paid-data.publicdata.landregistry.gov.uk/pp-uprn-lookup-jul-2026.csv
  Transaction unique identifier and UPRN Look Up Table — two fields, no header: the 38-character braced identifier, and a UPRN.



## Step 3 — Download, and fingerprint on the way in

Three things get recorded for each file beyond the bytes themselves:

- a **SHA-256**, so Silver can refuse to run if Bronze has been touched;
- the server's **`Last-Modified`** header, which is the only version marker
  these files carry — the URL never changes and the contents are replaced;
- a **line count**, done by counting newlines rather than parsing, because
  counting isn't transforming.

I also check the bytes written against `Content-Length`. A truncated download
that silently becomes Bronze is the kind of thing you discover three layers
later.

In [4]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def line_count(path):
    """Count records without parsing. Handles a missing trailing newline."""
    count, last = 0, b""
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            count += chunk.count(b"\n")
            last = chunk[-1:] or last
    return count + (1 if last and last != b"\n" else 0)


def fetch(filename, description):
    url = f"{BASE_URL}/{filename}"
    dest = BRONZE_DIR / filename
    with requests.get(url, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        headers = resp.headers
        with dest.open("wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    size = dest.stat().st_size
    declared = headers.get("Content-Length")
    if declared is not None and int(declared) != size:
        raise IOError(f"{filename}: truncated — declared {declared}, wrote {size}")

    return {
        "filename": filename,
        "source_url": url,
        "description": description,
        "http_last_modified": headers.get("Last-Modified"),
        "http_etag": headers.get("ETag"),
        "content_type": headers.get("Content-Type"),
        "size_bytes": size,
        "line_count": line_count(dest),
        "sha256": sha256(dest),
    }


files = [fetch(name, desc) for name, desc in SOURCES.items()]

for f in files:
    print(f"{f['filename']}")
    print(f"  {f['size_bytes']:>12,} bytes | {f['line_count']:>8,} lines")
    print(f"  last-modified: {f['http_last_modified']}")
    print(f"  sha256:        {f['sha256'][:32]}...\n")

pp-monthly-update-new-version.csv
    17,735,671 bytes |  101,600 lines
  last-modified: Fri, 28 Aug 2026 05:12:46 GMT
  sha256:        f92fff441ecbeba0f4891524fc968c8d...

pp-uprn-lookup-jul-2026.csv
     5,169,801 bytes |   94,112 lines
  last-modified: Fri, 28 Aug 2026 05:12:48 GMT
  sha256:        1d05becac589365a0f87b81f06b9a79c...



Both files carry the same `Last-Modified` — Friday 28 August 2026 at 05:12 GMT.
That's the release moment, and it's the closest thing to a version number either
file has.

101,600 lines of Price Paid Data against 94,112 lines of look-up. The gap is
already visible before I've joined anything.

## Step 4 — Is the look-up about this month, or about all thirty years?

This is the question that decides whether anything downstream means anything.

HMLR say the look-up "applies to monthly Price Paid Data published from 28
August 2026 onwards". That could mean it covers the transactions in *this
month's release*, or that it's a growing table that happens to have started
now. If I guessed wrong and joined it to the complete file, I'd get a match rate
near zero that measures nothing but the forward-only policy.

Rather than infer it from the wording, I can just check: does every identifier
in the look-up appear in this month's Price Paid file?

In [5]:
monthly_ids = set(
    pd.read_csv(BRONZE_DIR / "pp-monthly-update-new-version.csv",
                header=None, usecols=[0], dtype=str)[0]
)
lookup_ids = set(
    pd.read_csv(BRONZE_DIR / "pp-uprn-lookup-jul-2026.csv",
                header=None, usecols=[0], dtype=str)[0]
)

print(f"identifiers in the monthly file : {len(monthly_ids):>7,}")
print(f"identifiers in the look-up      : {len(lookup_ids):>7,}")
print(f"look-up ids found in monthly     : {len(lookup_ids & monthly_ids):>7,}")
print(f"look-up ids NOT in monthly       : {len(lookup_ids - monthly_ids):>7,}")
print(f"monthly ids with no look-up row  : {len(monthly_ids - lookup_ids):>7,}")

identifiers in the monthly file : 101,600
identifiers in the look-up      :  94,112
look-up ids found in monthly     :  94,112
look-up ids NOT in monthly       :       0
monthly ids with no look-up row  :   7,488


Every one of the 94,112 look-up identifiers is in this month's Price Paid file,
and **none** fall outside it. The look-up's universe is exactly this release.

So the denominator for any match rate is the 101,600-row monthly file — not the
complete file, and not some growing historical table. That's settled now, in
Bronze, rather than being assumed in Gold.

## Step 5 — Something to reconcile against

The whole pipeline is worth nothing if my extraction is quietly wrong, so I want
to reproduce a figure HMLR publish themselves and show it matches.

The problem: HMLR publish **no official match rate**, so there's nothing to check
the headline finding against. What they do publish is the same Price Paid Data
through a completely separate channel — a SPARQL triplestore at
`landregistry.data.gov.uk`. Same publisher, same records, different delivery
pipeline. If my CSV extraction and their triplestore agree on a transaction
count, that's real evidence and not a tautology.

One wrinkle. The monthly file is a **delta**, not a snapshot — it carries
whatever HMLR added or amended this cycle, with transfer dates running back to
1995. So for any transfer month except the newest, the triplestore holds this
release's rows *plus* rows published in earlier releases, and the two must
diverge. Only the newest transfer month is wholly contained in one release.

I query the newest month as the target, and three earlier months as controls —
so the record shows the divergence and proves the match is mechanism rather than
luck. These take a couple of minutes each.

In [6]:
import calendar

SPARQL_ENDPOINT = "https://landregistry.data.gov.uk/landregistry/query"
TARGET_MONTH = "2026-07"
CONTROL_MONTHS = ["2026-06", "2026-05", "2025-09"]

QUERY_TEMPLATE = """PREFIX ppi: <http://landregistry.data.gov.uk/def/ppi/>
SELECT (COUNT(*) AS ?n) WHERE {{
  ?t ppi:transactionDate ?d .
  FILTER(?d >= "{first}"^^<http://www.w3.org/2001/XMLSchema#date>
      && ?d <= "{last}"^^<http://www.w3.org/2001/XMLSchema#date>)
}}"""


def count_transactions(ym):
    year, month = (int(p) for p in ym.split("-"))
    first, last = f"{ym}-01", f"{ym}-{calendar.monthrange(year, month)[1]:02d}"
    query = QUERY_TEMPLATE.format(first=first, last=last)
    resp = requests.get(SPARQL_ENDPOINT, params={"query": query},
                        headers={"Accept": "application/sparql-results+json"},
                        timeout=300)
    resp.raise_for_status()
    count = int(resp.json()["results"]["bindings"][0]["n"]["value"])
    print(f"  {ym}: {count:,}")
    return {"transfer_month": ym, "count": count, "query": query}


print("HMLR linked data, transactions by transfer month:")
target = count_transactions(TARGET_MONTH)
controls = [count_transactions(ym) for ym in CONTROL_MONTHS]

HMLR linked data, transactions by transfer month:
  2026-07: 22,835
  2026-06: 46,025
  2026-05: 46,381
  2025-09: 80,340


22,835 transfers dated July 2026 according to HMLR's own triplestore. Silver
will check my extraction against that number.

The controls are the interesting part of the record: 46,025 for June and 46,381
for May, both far more than this release contains, because most of those months
shipped in earlier releases. That's exactly what a delta file should look like.

While I was here I also learned something that would have cost me an afternoon
later: the triplestore exposes the identifier as a bare 36-character GUID typed
as `ppi:TransactionIdDatatype`, while both CSVs carry the 38-character **braced**
form. A plain string match returns nothing. Noted in `docs/sources.md`.

## Step 6 — Write the manifest

Everything I've learned about these files goes next to them. Two JSON sidecars:
the provenance manifest, and the reconciliation reference with each SPARQL query
stored verbatim so the gate can be re-run rather than trusted.

I also record HMLR's Transaction Data figure for July — 103,398 "transactions
for value" — but explicitly **not** as a reconciliation target. It counts
applications *completed* in the month; my file is a delta of adds, changes and
deletions spanning three decades. Different universes. Forcing them to agree
would be manufacturing a match.

In [7]:
metadata = {
    "publisher": "HM Land Registry",
    "release_slug": RELEASE_SLUG,
    "publication_date": "2026-08-28",
    "data_month": "2026-07",
    "publication_schedule": "20th working day of each month",
    "licence": "Open Government Licence v3.0",
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "retention_note": (
        "HMLR retains only the latest version of each of these files. Both URLs "
        "are stable and overwritten in place, so this Bronze copy is not "
        "reproducible from the source after the next release."
    ),
    "files": files,
}
(BRONZE_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

reference = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "linked_data": {
        "endpoint": SPARQL_ENDPOINT,
        "publisher": "HM Land Registry",
        "licence": "Open Government Licence v3.0",
        "reconciliation_month": TARGET_MONTH,
        "target": target,
        "controls": controls,
    },
    "transaction_data_reference": {
        "source_url": "https://www.gov.uk/government/news/july-2026-transaction-data",
        "published_date": "2026-08-21",
        "period": "July 2026",
        "coverage": "England and Wales",
        "basis": "customer applications completed in the month, by completion date",
        "transactions_for_value": 103398,
        "total_applications_completed": 2134898,
        "use": "context only — not a reconciliation target",
    },
}
(BRONZE_DIR / "reconciliation_reference.json").write_text(json.dumps(reference, indent=2))

for path in sorted(BRONZE_DIR.iterdir()):
    print(f"{path.name:<40} {path.stat().st_size:>12,} bytes")

metadata.json                                   1,780 bytes
pp-monthly-update-new-version.csv          17,735,671 bytes
pp-uprn-lookup-jul-2026.csv                 5,169,801 bytes
reconciliation_reference.json                   2,196 bytes


**Bronze is done.** Two raw CSVs exactly as served, two JSON sidecars, and
nothing altered.

What I know going into Silver: the denominator is 101,600 rows; 94,112 of them
have a look-up row and 7,488 don't; and I have a number from HMLR — 22,835 — that
my extraction has to reproduce before any of the analysis is allowed to count.